# 13 用 PyTorch 观察 Attention 梯度

前两课已经建立了 Attention 的反向传播路线和核心公式。

这一课不训练完整 Transformer，而是构造一个只有 3 个 token 的迷你 Self-Attention，观察一次 `backward()` 究竟产生了哪些梯度。

本课目标：

- 看见 $W_Q$、$W_K$、$W_V$ 都能收到梯度；
- 看见中间量 $A$、$S$、$Q$、$K$、$V$ 的梯度；
- 用矩阵公式核对 PyTorch 自动求导结果；
- 理解 `backward()` 计算梯度，但不会自动更新参数。

## 1. 固定一个最小形状

为了让所有矩阵都能直接查看，设定：

$$
N=3,\qquad D=4,\qquad d_k=2,\qquad d_v=3.
$$

对应形状：

$$
\begin{aligned}
X&:3\times4,\\
W_Q,W_K&:4\times2,\\
W_V&:4\times3,\\
Q,K&:3\times2,\\
V,O&:3\times3,\\
S,A&:3\times3.
\end{aligned}
$$

In [ ]:
import math
import torch

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

N, D, d_k, d_v = 3, 4, 2, 3

X = torch.randn(N, D, requires_grad=True)
W_Q = torch.randn(D, d_k, requires_grad=True)
W_K = torch.randn(D, d_k, requires_grad=True)
W_V = torch.randn(D, d_v, requires_grad=True)

X.shape, W_Q.shape, W_K.shape, W_V.shape

## 2. 手动写出 Attention 前向传播

这里故意不调用封装好的 `MultiheadAttention`，因为我们希望看清每一个中间节点。

对非叶子张量调用 `retain_grad()`，是为了在反向传播后保留它们的 `.grad`，方便观察。正常训练时通常不需要这样做。

In [ ]:
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

S = (Q @ K.T) / math.sqrt(d_k)
A = torch.softmax(S, dim=-1)
O = A @ V

for tensor in (Q, K, V, S, A, O):
    tensor.retain_grad()

print('Q:', Q.shape)
print('K:', K.shape)
print('V:', V.shape)
print('S:', S.shape)
print('A:', A.shape)
print('O:', O.shape)
print('每行注意力权重之和:', A.sum(dim=-1))

## 3. 构造一个简单损失

这里不做真实分类，只让所有输出元素尽量接近 0：

$$
\mathcal{L}=\frac{1}{Nd_v}\sum_{i,j}O_{ij}^{2}.
$$

这个任务没有实际意义，但它可以提供明确的梯度，非常适合观察反向传播。

In [ ]:
loss = O.square().mean()
print('loss =', loss.item())

loss.backward()

## 4. 先确认三组参数都收到了梯度

只要 `.grad` 不是 `None`，并且梯度范数通常不为 0，就说明损失可以沿计算图到达这组参数。

注意：某次梯度恰好出现 0 并不一定代表计算图断开，也可能是当前数据和参数造成的数学结果。

In [ ]:
for name, tensor in [('W_Q', W_Q), ('W_K', W_K), ('W_V', W_V), ('X', X)]:
    print(f'{name:>3} grad shape = {tuple(tensor.grad.shape)}, norm = {tensor.grad.norm().item():.6f}')

## 5. 查看完整的梯度路线

下面依次查看输出、注意力权重、分数、QKV 的梯度形状。

它们应当分别与 $O$、$A$、$S$、$Q$、$K$、$V$ 自己的形状一致。

In [ ]:
for name, tensor in [('O', O), ('A', A), ('S', S), ('Q', Q), ('K', K), ('V', V)]:
    print(f'{name} shape = {tuple(tensor.shape)}, grad shape = {tuple(tensor.grad.shape)}')

print('\nA.grad =')
print(A.grad)
print('\nS.grad =')
print(S.grad)

## 6. 核对 $O=AV$ 的两条梯度公式

上一课推导了：

$$
G_A=G_OV^{\top},\qquad G_V=A^{\top}G_O.
$$

现在用矩阵乘法手动计算，再与 PyTorch 的结果比较。`detach()` 表示这里只取数值，不再创建新的求导路线。

In [ ]:
expected_G_A = O.grad @ V.detach().T
expected_G_V = A.detach().T @ O.grad

print('G_A 是否一致:', torch.allclose(A.grad, expected_G_A, atol=1e-6))
print('G_V 是否一致:', torch.allclose(V.grad, expected_G_V, atol=1e-6))

## 7. 观察 Softmax 梯度的一行和

Softmax 对整行分数同时加上相同常数不敏感，因此每一行 $G_S$ 的元素之和应当接近 0：

$$
\sum_j(G_S)_{ij}\approx0.
$$

由于浮点数计算存在误差，结果可能不是严格的 0，而是非常小的数。

In [ ]:
print('S.grad 每一行的和:')
print(S.grad.sum(dim=-1))

## 8. 核对 $S=QK^{\top}/\sqrt{d_k}$ 的梯度

上一课得到：

$$
G_Q=\frac{G_SK}{\sqrt{d_k}},\qquad G_K=\frac{G_S^{\top}Q}{\sqrt{d_k}}.
$$

In [ ]:
expected_G_Q = (S.grad @ K.detach()) / math.sqrt(d_k)
expected_G_K = (S.grad.T @ Q.detach()) / math.sqrt(d_k)

print('G_Q 是否一致:', torch.allclose(Q.grad, expected_G_Q, atol=1e-6))
print('G_K 是否一致:', torch.allclose(K.grad, expected_G_K, atol=1e-6))

## 9. 核对参数梯度

对于 $Q=XW_Q$，参数梯度为：

$$
G_{W_Q}=X^{\top}G_Q.
$$

$W_K$、$W_V$ 同理。

In [ ]:
expected_G_WQ = X.detach().T @ Q.grad
expected_G_WK = X.detach().T @ K.grad
expected_G_WV = X.detach().T @ V.grad

print('G_WQ 是否一致:', torch.allclose(W_Q.grad, expected_G_WQ, atol=1e-6))
print('G_WK 是否一致:', torch.allclose(W_K.grad, expected_G_WK, atol=1e-6))
print('G_WV 是否一致:', torch.allclose(W_V.grad, expected_G_WV, atol=1e-6))

## 10. 核对输入 $X$ 的三路梯度汇总

输入 $X$ 同时生成 Q、K、V，因此：

$$
G_X=G_QW_Q^{\top}+G_KW_K^{\top}+G_VW_V^{\top}.
$$

In [ ]:
expected_G_X = (
    Q.grad @ W_Q.detach().T
    + K.grad @ W_K.detach().T
    + V.grad @ W_V.detach().T
)

print('G_X 是否等于三条路线之和:', torch.allclose(X.grad, expected_G_X, atol=1e-6))

## 11. `backward()` 之后参数还没有更新

执行 `loss.backward()` 后：

- $W_Q$、$W_K$、$W_V$ 的数值没有变化；
- 它们的 `.grad` 中保存了修改方向；
- 只有执行优化器的 `step()`，参数才会真正更新。

训练循环的基本职责仍然是：

$$
\text{清空梯度}\rightarrow\text{前向传播}\rightarrow\text{计算损失}\rightarrow\text{反向传播}\rightarrow\text{更新参数}.
$$

Attention 只是计算图更复杂，并没有改变神经网络训练的基本流程。

## 12. 本节小结

1. Attention 中的 $W_Q$、$W_K$、$W_V$ 都可以从同一个损失收到梯度。
2. `retain_grad()` 可以帮助我们观察非叶子中间张量的梯度。
3. PyTorch 对 $O=AV$ 的自动求导结果与 $G_A=G_OV^{\top}$、$G_V=A^{\top}G_O$ 一致。
4. PyTorch 对分数矩阵的求导结果与 $G_Q=G_SK/\sqrt{d_k}$、$G_K=G_S^{\top}Q/\sqrt{d_k}$ 一致。
5. $X$ 的梯度确实是 Q、K、V 三条分支返回梯度之和。
6. `backward()` 只计算梯度，优化器才负责更新参数。

## 13. 自测问题

1. 为什么本课不直接使用封装好的 `MultiheadAttention`？
2. 为什么要对 $Q$、$K$、$V$、$S$、$A$、$O$ 调用 `retain_grad()`？
3. 如何用 PyTorch 核对 $G_A=G_OV^{\top}$？
4. 为什么 $S$ 的每一行梯度之和接近 0？
5. 哪段实验说明了 $1/\sqrt{d_k}$ 也参与反向传播？
6. 哪段实验说明了 $X$ 会汇总三条分支的梯度？
7. 执行 `loss.backward()` 后，参数是否已经更新？